# Feature Engineering

This notebook engineers rich features from the cleaned and preprocessed datasets.

Inputs (data/processed/):
- movies_integrated.csv (from 03_data_preprocessing.ipynb)
- train_ratings.csv, test_ratings.csv

Outputs (data/processed/):
- movie_engineered.csv — movie-level features (popularity, temporal, content signals)
- user_engineered.csv — user-level features (activity, genre preferences, rating behaviour)
- movie_features_selected.csv — subset after feature selection
- user_features_selected.csv — subset after feature selection

Run this notebook after 03_data_preprocessing.ipynb and before 05_popularity_baseline.ipynb.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

import sys
sys.path.append("..")

from src.data.data_loader import DataLoader
from src.data.data_preprocessor import DataPreprocessor
from src.features.feature_engineering import FeatureEngineer
from src.features.feature_selection import FeatureSelector

print("Libraries loaded.")

## Load Preprocessed Data

In [ ]:
loader = DataLoader()
preprocessor = DataPreprocessor()

print("Loading integrated movies...")
movies = loader.load_movies_integrated()
print(f"  movies_integrated: {movies.shape}")

print("Loading train/test split...")
train_ratings, test_ratings = preprocessor.load_train_test_split()
print(f"  train_ratings: {train_ratings.shape}")
print(f"  test_ratings:  {test_ratings.shape}")

## Engineer Movie Features

Movie-level features computed from the training set:
- Popularity score (Bayesian weighted rating)
- Temporal features (release year, decade)
- Vote statistics (average, count bins)
- Genre one-hot encoding

In [ ]:
engineer = FeatureEngineer()

print("Engineering movie features...")
movie_features = engineer.engineer_movie_features(
    train_ratings=train_ratings,
    movies_integrated=movies,
)
print(f"  movie_features shape: {movie_features.shape}")
print(f"  Columns: {list(movie_features.columns[:10])} ... ({len(movie_features.columns)} total)")

## Engineer User Features

User-level features computed from the training set:
- Activity level (total ratings, rating frequency)
- Genre preferences (weighted average rating per genre)
- Rating behaviour (mean, std, tendency to rate high/low)

In [ ]:
print("Engineering user features (this may take a few minutes for 162K users)...")
user_features = engineer.engineer_user_features(
    train_ratings=train_ratings,
    movies_integrated=movies,
)
print(f"  user_features shape: {user_features.shape}")
print(f"  Columns: {list(user_features.columns[:10])} ... ({len(user_features.columns)} total)")

## Feature Selection

Reduce dimensionality by selecting the most informative features using variance thresholding and correlation analysis.

In [ ]:
selector = FeatureSelector()

print("Selecting movie features...")
movie_features_selected = selector.select_features(movie_features, target_col=None)
print(f"  {movie_features.shape[1]} -> {movie_features_selected.shape[1]} features")

print("Selecting user features...")
user_features_selected = selector.select_features(user_features, target_col=None)
print(f"  {user_features.shape[1]} -> {user_features_selected.shape[1]} features")

## Save Engineered Features

In [ ]:
engineer.save_engineered_features(
    user_engineered=user_features,
    movie_engineered=movie_features,
)
engineer.save_features(
    user_features=user_features_selected,
    movie_features=movie_features_selected,
)
print("Feature files saved to data/processed/")
print("Next: run 05_popularity_baseline.ipynb")

## Feature Summary

| Feature Group | Count | Description |
|---|---|---|
| Movie popularity | ~5 | Bayesian score, vote count bins, rating percentile |
| Temporal | ~4 | Release year, decade, recency |
| Genre encoding | ~20 | One-hot per genre |
| User activity | ~6 | Total ratings, active months, avg interval |
| User genre prefs | ~20 | Weighted avg rating per genre |
| User behaviour | ~5 | Rating mean, std, high/low bias |